# Multi-Dimensional UMAP Export for Web Viewer

This notebook doubles as a tutorial for exporting deterministic 1D/2D/3D embeddings into the Cellucid WebGL viewer bundles.

> This variant is pre-configured for the **Human Developmental Cell Atlas (HDCA)** dataset; all source files live on the compute cluster at `/lustre/groups/ml01/workspace/kemal.inecik/hdca/`. Tweak the configuration cell if your files live elsewhere.

**In this walkthrough you will**
- configure dataset roots once so the same notebook runs without editing paths elsewhere.
- verify that expression counts (`X`) and the latent representation (`obsm['latent']`) are present before running anything expensive.
- remap `var.index` from Ensembl gene IDs to HGNC gene symbols so the viewer displays human-readable gene names.
- recompute missing UMAP dimensions only when necessary while reusing a single neighbor graph for perfect alignment across 1D/2D/3D.
- hydrate metadata + expressions and feed everything into `cellucid.prepare`.
- validate the generated binary assets and manifests before handing them to the frontend.

Each section calls out why the code exists so you can adapt the pattern to your own datasets.


## Environment

These setup helpers make the notebook location-agnostic: run it from the repo root, from `notebooks/`, or from VS Code and the imports/paths will still resolve.


In [1]:
1

1

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys
import gc
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad

HERE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(HERE)
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

sc.settings.verbosity = 3

/home/icb/kemal.inecik/tools/apps/mamba/envs/apidip_dev_env/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Configuration

Keep all project-specific paths and knobs together so rerunning exports becomes a one-cell edit exercise.

**Key differences from smaller datasets:**
- `SOURCE_FILE` lives on lustre (the server) rather than in the project `data/` tree — it is too large to copy locally.
- `EXPERIMENT_FILE` (the intermediate h5ad that stores computed UMAP embeddings) is written alongside the source file on lustre to avoid re-computation across sessions.
- Gene names: `var.index` in the source file contains Ensembl IDs; we remap to HGNC symbols via `var['hgnc']` before export.


In [4]:
# Dataset slug — used to name the export directory
DATASET_NAME = "hdca"

# Source file: integration h5ad produced by the scVI/scANVI unification pipeline.
# Contains expression counts in X and the latent representation in obsm['latent'].
SOURCE_FILE = Path(
    "/lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/"
    "20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration.h5ad"
)

# Intermediate file: written once UMAP embeddings have been computed.
# Saved next to the source file so we can skip recomputation on subsequent runs.
EXPERIMENT_FILE = SOURCE_FILE.parent / (
    SOURCE_FILE.stem + "_umap.h5ad"
)

# Export destination consumed by the Cellucid WebGL viewer
EXPORT_DIR = PROJECT_ROOT.parent / "cellucid-datasets" / "exports" / DATASET_NAME

print(f"SOURCE_FILE  : {SOURCE_FILE}")
print(f"EXPERIMENT_FILE: {EXPERIMENT_FILE}")
print(f"EXPORT_DIR   : {EXPORT_DIR}")

SOURCE_FILE  : /lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration.h5ad
EXPERIMENT_FILE: /lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration_umap.h5ad
EXPORT_DIR   : /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca


## Verify source data

Open the file in backed (memory-mapped) mode to confirm that:
- `X` (expression counts) is present and the right shape, and
- `obsm['latent']` (the scVI latent space) is present.

Both are required downstream — counts for the quantized gene-expression export, and the latent space for computing neighbors and centroid statistics.


In [ ]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

adata_preview = ad.read_h5ad(SOURCE_FILE, backed="r")
print(adata_preview)
print()

# --- Expression counts ---
if adata_preview.X is None:
    raise ValueError("X is None — expression counts are missing from the source file.")
print(f"✓ X present: shape {adata_preview.X.shape}, dtype {adata_preview.X.dtype}")

# --- Latent space ---
LATENT_KEY = "latent"
if LATENT_KEY not in adata_preview.obsm:
    raise KeyError(
        f"obsm['{LATENT_KEY}'] not found. Available keys: {list(adata_preview.obsm.keys())}"
    )
print(f"✓ obsm['{LATENT_KEY}'] present: shape {adata_preview.obsm[LATENT_KEY].shape}")

# --- var index (gene IDs before remapping) ---
print(f"\nvar.index (Ensembl IDs, first 5): {adata_preview.var.index[:6].tolist()}")
print(f"var['hgnc'] (gene symbols, first 5): {adata_preview.var['hgnc'][:6].tolist()}")
print(np.all(adata_preview.var.index==adata_preview.var['hgnc']))

# --- Observation metadata ---
print(f"\nobs columns: {adata_preview.obs.columns.tolist()}")
if "LVL0" in adata_preview.obs:
    print("\nLVL0 value counts:")
    for label, count in adata_preview.obs["LVL0"].value_counts().items():
        print(f"  {label}: {count:,}")

adata_preview.file.close()
del adata_preview

AnnData object with n_obs × n_vars = 3652086 × 12288 backed at '/lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration.h5ad'
    obs: 'handle_anndata', 'study', 'sample_ID', 'organ', 'age', 'cell_type', 'lane_ID', 'author_batch', 'institute', 'study_PI', 'doi', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'anatomical_region', 'anatomical_region_level_2', 'sex', 'sex_inferred', 'subject_type', 'sample_status', 'sample_cultured', 'protocol_tissue_dissociation', 'cell_enrichment', 'library_platform', 'strand_sequence', 'sequencing_platform', 'reads_processing', 'biological_unit', 'reference_genome', 'reference_genome_ensembl_release', 'concatenated_integration_covariates', 'original_author_annotation', 'LVL3', 'LVL2', 'LVL1', 'LVL0', '_scvi_batch', '_scvi_labels', 'split'
    var: 'hgnc'
    uns: '_scvi_man

## UMAP parameters

All tunable knobs live here.  Change them only when you explicitly want a new layout — keeping them stable ensures reproducible exports between releases.


In [8]:
# kNN graph parameters (shared across all UMAP dimensionalities)
n_neighbors = 15
min_dist = 0.5
RANDOM_SEED = 0

# Storage keys for each dimensionality we care about.
UMAP_DIMENSION_KEYS = {
    1: "X_umap_1d",
    2: "X_umap_2d",
    3: "X_umap_3d",
    # 4: "X_umap_4d",  # Reserved for future expansion
}


def compute_umap_embedding(adata_source, n_components: int, min_dist: float, random_state: int) -> np.ndarray:
    """Compute a UMAP embedding with the provided dimensionality without mutating the source AnnData."""
    neighbors_params = adata_source.uns.get("neighbors", {}).get("params", {})
    use_rep = neighbors_params.get("use_rep", None)

    adata_temp = ad.AnnData(
        obs=adata_source.obs[[]],
        obsp={
            "connectivities": adata_source.obsp["connectivities"],
            "distances": adata_source.obsp["distances"],
        },
    )

    if use_rep is not None and use_rep in adata_source.obsm:
        adata_temp.obsm[use_rep] = adata_source.obsm[use_rep]

    adata_temp.uns["neighbors"] = adata_source.uns["neighbors"].copy()
    sc.tl.umap(adata_temp, n_components=n_components, min_dist=min_dist, random_state=random_state)

    embedding = adata_temp.obsm["X_umap"].copy()
    del adata_temp
    return embedding

## Deterministic Embedding Strategy

- **Stable random seed (`RANDOM_SEED`)** keeps layouts reproducible between releases, which is critical when comparing viewer builds, spotting regression diffs, or debugging quantization artifacts.
- **Stable kNN graph for all dimensions** means `sc.pp.neighbors` runs once and every 1D/2D/3D embedding encodes the exact same neighbor relationships; cross-dimensional brushing stays intuitive and centroid statistics stay comparable.
- **Shared latent representation (`latent`)** ensures the centroids and connectivities exported later line up with whatever representation was used in training; no silent drift between the viewer and the model.
- **Legacy `adata.obsm['X_umap']` alias** mirrors the 3D embedding under the historical key so older ingestion scripts and viewer builds continue to work even though the tutorial now emits explicit multi-dimensional files.
- **Backed AnnData checks** let us peek into the `.h5ad` file without loading it fully and skip recomputation when the embeddings are already up to date.
- **EXPERIMENT_FILE on lustre** means the ~3.65 M-cell UMAP result is stored next to its source rather than in the project tree, keeping the repository lean.

Tweak the parameters in the previous cell only when you explicitly want to generate alternative deterministic layouts.


In [9]:
def umap_dimensions_present(exp_file: Path, dim_keys: dict) -> tuple:
    """Return whether each required UMAP embedding is stored in exp_file plus the missing dimensions."""
    if exp_file is None or not exp_file.exists():
        return False, list(dim_keys.keys())

    backed = ad.read_h5ad(exp_file, backed="r")
    try:
        available = set(backed.obsm_keys())
    finally:
        backed.file.close()
    missing = [dim for dim, key in dim_keys.items() if key not in available]
    return len(missing) == 0, missing


def ensure_umap_embeddings():
    """Compute multi-dimensional UMAP embeddings only when they are absent on disk."""
    ready, missing_dims = umap_dimensions_present(EXPERIMENT_FILE, UMAP_DIMENSION_KEYS)

    if ready:
        print(
            f"✓ {EXPERIMENT_FILE.name} already stores "
            f"{', '.join(f'{dim}D' for dim in UMAP_DIMENSION_KEYS)} embeddings."
        )
        return

    missing_msg = ", ".join(f"{dim}D" for dim in missing_dims) if missing_dims else "all required"
    if EXPERIMENT_FILE.exists():
        print(f"Updating {EXPERIMENT_FILE.name}: missing {missing_msg} embeddings.")
    else:
        print(f"{EXPERIMENT_FILE} does not exist yet. Computing full multi-dimensional embeddings.")

    if not SOURCE_FILE.exists():
        raise FileNotFoundError(f"Source file not found: {SOURCE_FILE}")

    print(f"Loading {SOURCE_FILE.name} into memory (~3.65 M cells × 12 288 genes — this will take a while)...")
    adata = ad.read_h5ad(SOURCE_FILE)

    # Remap var.index from Ensembl gene IDs to HGNC gene symbols before saving.
    # The 'hgnc' column contains human-readable gene names (no NaN values in this dataset).
    print(f"Remapping var.index: Ensembl IDs → HGNC symbols via var['hgnc'] ...")
    adata.var.index = adata.var["hgnc"].values
    adata.var.index.name = "gene_name"

    print(f"Computing neighbors on obsm['{LATENT_KEY}'] (shared graph for all UMAP dimensionalities)...")
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, random_state=RANDOM_SEED, use_rep=LATENT_KEY)

    for n_dim, key in UMAP_DIMENSION_KEYS.items():
        print(f"Computing {n_dim}D UMAP → {key}")
        adata.obsm[key] = compute_umap_embedding(
            adata, n_components=n_dim, min_dist=min_dist, random_state=RANDOM_SEED
        )

    if 3 in UMAP_DIMENSION_KEYS:
        # Keep a 3D copy under "X_umap" because legacy viewer builds still expect this key
        # even though newer ones read the explicit multi-dimensional files.
        adata.obsm["X_umap"] = adata.obsm[UMAP_DIMENSION_KEYS[3]].copy()

    print("UMAP embeddings computed:")
    for n_dim, key in UMAP_DIMENSION_KEYS.items():
        shape = adata.obsm[key].shape
        print(f"  {key}: {shape}")

    EXPERIMENT_FILE.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(EXPERIMENT_FILE)
    print(f"Saved updated embeddings to {EXPERIMENT_FILE}")

    del adata
    gc.collect()


ensure_umap_embeddings()

/lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration_umap.h5ad does not exist yet. Computing full multi-dimensional embeddings.
Loading 20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration.h5ad into memory (~3.65 M cells × 12 288 genes — this will take a while)...
Remapping var.index: Ensembl IDs → HGNC symbols via var['hgnc'] ...
Computing neighbors on obsm['latent'] (shared graph for all UMAP dimensionalities)...
computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:16:46)
Computing 1D UMAP → X_umap_1d
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (1:00:35)
Computing 2D UMAP → X_umap_2d
computing UMAP
    finished: added


/home/icb/kemal.inecik/tools/apps/mamba/envs/apidip_dev_env/lib/python3.11/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (1:18:58)
UMAP embeddings computed:
  X_umap_1d: (3652086, 1)
  X_umap_2d: (3652086, 2)
  X_umap_3d: (3652086, 3)
Saved updated embeddings to /lustre/groups/ml01/workspace/kemal.inecik/hdca/temp/models/20240613_152640100145_gpusrv44.scidom.de_2367215_unification_union_20241216_hvg-intersection_integration_umap.h5ad


## Load UMAP run

The previous step guarantees that the experiment file exists and houses every required UMAP dimension. Load it now and double-check which embeddings are present.


In [10]:
if not EXPERIMENT_FILE.exists():
    raise FileNotFoundError(
        f"UMAP file not found at {EXPERIMENT_FILE}. Run the cell above first."
    )

adata = ad.read_h5ad(EXPERIMENT_FILE)

# Coerce age to numeric in case it was stored as strings
if "age" in adata.obs:
    adata.obs["age"] = pd.to_numeric(adata.obs["age"], errors="coerce")

# Drop internal scVI bookkeeping columns that are not meaningful to end users
drop_columns = [col for col in ("_scvi_batch", "_scvi_labels") if col in adata.obs]
if drop_columns:
    adata.obs = adata.obs.drop(columns=drop_columns)

legacy_umap_key = "X_umap"  # 3D default used by legacy exports/viewers
available_umaps = {}
for dim, key in UMAP_DIMENSION_KEYS.items():
    dim_label = f"{dim}d"
    if key in adata.obsm:
        available_umaps[dim_label] = adata.obsm[key]
        print(f"✓ Found {key}: shape {adata.obsm[key].shape}")
    else:
        print(f"✗ Missing {key}")

if not available_umaps:
    if legacy_umap_key in adata.obsm:
        print(f"Using legacy {legacy_umap_key}")
        available_umaps['3d'] = adata.obsm[legacy_umap_key]
    else:
        raise KeyError("No UMAP embeddings found in adata.obsm")

print(f"\nAvailable dimensions: {list(available_umaps.keys())}")
adata

✓ Found X_umap_1d: shape (3652086, 1)
✓ Found X_umap_2d: shape (3652086, 2)
✓ Found X_umap_3d: shape (3652086, 3)

Available dimensions: ['1d', '2d', '3d']


AnnData object with n_obs × n_vars = 3652086 × 12288
    obs: 'handle_anndata', 'study', 'sample_ID', 'organ', 'age', 'cell_type', 'lane_ID', 'author_batch', 'institute', 'study_PI', 'doi', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'anatomical_region', 'anatomical_region_level_2', 'sex', 'sex_inferred', 'subject_type', 'sample_status', 'sample_cultured', 'protocol_tissue_dissociation', 'cell_enrichment', 'library_platform', 'strand_sequence', 'sequencing_platform', 'reads_processing', 'biological_unit', 'reference_genome', 'reference_genome_ensembl_release', 'concatenated_integration_covariates', 'original_author_annotation', 'LVL3', 'LVL2', 'LVL1', 'LVL0', 'split'
    var: 'hgnc'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'neighbors', 'rank_genes_groups'
    obsm: 'X_umap', 'X_umap_1d', 'X_umap_2d', 'X_umap_3d', 'latent'
    obsp: 'connectivities', 'distances'

## Quick UMAP stats

Lightweight sanity check on all loaded UMAP embeddings (1D, 2D, 3D).


In [11]:
# Stats for all available UMAP dimensions
umap_stats = {}
for dim, coords in available_umaps.items():
    umap_stats[dim] = {
        "shape": coords.shape,
        "mean": coords.mean(axis=0).tolist(),
        "std": coords.std(axis=0).tolist(),
        "min": coords.min(axis=0).tolist(),
        "max": coords.max(axis=0).tolist(),
    }

print(f"UMAP stats for {adata.n_obs:,} cells:")
for dim, stats in umap_stats.items():
    print(f"\n{dim.upper()}:")
    print(f"  Shape: {stats['shape']}")
    print(f"  Mean: {[f'{x:.3f}' for x in stats['mean']]}")
    print(f"  Std:  {[f'{x:.3f}' for x in stats['std']]}")

umap_stats

UMAP stats for 3,652,086 cells:

1D:
  Shape: (3652086, 1)
  Mean: ['9.764']
  Std:  ['15.402']

2D:
  Shape: (3652086, 2)
  Mean: ['9.626', '5.258']
  Std:  ['6.758', '7.430']

3D:
  Shape: (3652086, 3)
  Mean: ['4.956', '4.994', '5.030']
  Std:  ['4.106', '4.037', '4.137']


{'1d': {'shape': (3652086, 1),
  'mean': [9.764443397521973],
  'std': [15.401933670043945],
  'min': [-24.136295318603516],
  'max': [42.893531799316406]},
 '2d': {'shape': (3652086, 2),
  'mean': [9.62608814239502, 5.257988452911377],
  'std': [6.758121013641357, 7.429720401763916],
  'min': [-8.188597679138184, -12.192293167114258],
  'max': [27.13654327392578, 22.3264102935791]},
 '3d': {'shape': (3652086, 3),
  'mean': [4.956271648406982, 4.993710994720459, 5.03004264831543],
  'std': [4.105706691741943, 4.037399768829346, 4.137169361114502],
  'min': [-6.016291618347168, -4.096835136413574, -5.227531433105469],
  'max': [15.448166847229004, 17.264394760131836, 15.014514923095703]}}

## Prepare expression data

The HDCA integration file stores raw expression counts directly in `X` alongside the latent space, so no separate "complete" AnnData file is needed.  We:
- confirm gene names are already remapped to HGNC symbols (done in `ensure_umap_embeddings`),
- normalize counts to 10 000 counts per cell and log1p-transform so the quantized export stays well-behaved,
- peek at a few non-zero values as a quick sanity check.

> **Memory note:** The full 3.65 M × 12 288 expression matrix is loaded here.  On a machine with ≥ 200 GB RAM this should fit; otherwise consider chunking or exporting from backed mode.


In [12]:
# Verify gene names on var.index are HGNC symbols (set during ensure_umap_embeddings)
print(f"var.index name : {adata.var.index.name}")
print(f"var.index (first 5): {adata.var.index[:5].tolist()}")

# Check for duplicate gene names — HGNC remapping should be unique but worth confirming
n_duplicates = adata.var.index.duplicated().sum()
if n_duplicates > 0:
    print(f"WARNING: {n_duplicates} duplicate gene names detected in var.index.")
    print("Duplicates:", adata.var.index[adata.var.index.duplicated()].tolist()[:10])
else:
    print(f"✓ All {adata.n_vars:,} gene names are unique.")

var.index name : gene_name
var.index (first 5): ['AC058791.1', 'AC062029.1', 'AC084809.2', 'C2orf15', 'CBWD3']
✓ All 12,288 gene names are unique.


In [13]:
# Normalize and log-transform expression counts in-place
# (adata.X is counts from the integration file — safe to transform)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata

normalizing counts per cell
    finished (0:00:12)


AnnData object with n_obs × n_vars = 3652086 × 12288
    obs: 'handle_anndata', 'study', 'sample_ID', 'organ', 'age', 'cell_type', 'lane_ID', 'author_batch', 'institute', 'study_PI', 'doi', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'anatomical_region', 'anatomical_region_level_2', 'sex', 'sex_inferred', 'subject_type', 'sample_status', 'sample_cultured', 'protocol_tissue_dissociation', 'cell_enrichment', 'library_platform', 'strand_sequence', 'sequencing_platform', 'reads_processing', 'biological_unit', 'reference_genome', 'reference_genome_ensembl_release', 'concatenated_integration_covariates', 'original_author_annotation', 'LVL3', 'LVL2', 'LVL1', 'LVL0', 'split'
    var: 'hgnc'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'neighbors', 'rank_genes_groups', 'log1p'
    obsm: 'X_umap', 'X_umap_1d', 'X_umap_2d', 'X_umap_3d', 'latent'
    obsp: 'connectivities', 'distances'

In [14]:
# Quick non-zero value preview on the first cell
row = adata.X[0]
dense_row = row.A.ravel() if hasattr(row, "A") else np.asarray(row).ravel()
non_zero_preview = dense_row[dense_row != 0][:5]
print(f"Non-zero log-normalized values (first cell, first 5): {non_zero_preview}")
non_zero_preview

Non-zero log-normalized values (first cell, first 5): [2.82621353 1.29387835 2.19061493 2.19061493 3.09897567]


array([2.82621353, 1.29387835, 2.19061493, 2.19061493, 3.09897567])

## Export for web viewer

`cellucid.prepare` handles the heavy lifting described in `src/cellucid/prepare_data.py`:
- quantizes continuous obs/var fields and expression matrices to keep payloads small,
- auto-picks compact categorical dtypes and gzips the resulting binaries, and
- emits dataset manifests (`dataset_identity.json`, `obs_manifest.json`, `var_manifest.json`) that the WebGL viewer reads at runtime.

The call below wires our multi-dimensional UMAPs plus metadata into that exporter.  Because `var.index` is now HGNC symbols, `var_gene_id_column=None` tells the exporter to use the index directly as gene identifiers.


In [15]:
from cellucid import prepare

In [16]:
prepare(
    # Multi-dimensional UMAP embeddings; missing keys evaluate to None and will be skipped
    X_umap_1d=available_umaps.get('1d'),
    X_umap_2d=available_umaps.get('2d'),
    X_umap_3d=available_umaps.get('3d'),
    # X_umap_4d is reserved for future development

    # Other data matrices (scVI latent space drives centroids/kNN reuse)
    latent_space=adata.obsm[LATENT_KEY],
    obs=adata.obs,
    var=adata.var,
    gene_expression=adata.X,
    connectivities=adata.obsp['connectivities'],

    # Export behavior knobs defined inline for clarity
    var_gene_id_column=None,  # var.index already contains HGNC gene symbols
    gene_identifiers=None,    # Export every gene; slice list here if needed
    centroid_outlier_quantile=0.90,  # Trim cells far from centroid when summarizing categories
    centroid_min_points=10,          # Require at least this many cells per centroid
    force=False,
    var_quantization=8,
    obs_continuous_quantization=8,
    obs_categorical_dtype="auto",
    compression=6,

    # Dataset identity metadata surfaced in dataset_identity.json
    out_dir=EXPORT_DIR,
    dataset_name=DATASET_NAME,
    dataset_description="Human Developmental Cell Atlas — multi-organ single-cell transcriptomics atlas of human development",
    source_name="HDCA",
    source_url="https://www.humancellatlas.org/"
)

Export Settings:
  Output directory: /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca
  Compression: gzip level 6
  Var (gene) quantization: 8-bit
  Obs continuous quantization: 8-bit
  Obs categorical dtype: auto
  Available dimensions: [1, 2, 3]
  Default dimension: 3D
  Coordinate normalization (per-dimension, aspect-ratio preserved):
    1D: range 67.03 → [-1, 1]
    2D: range 35.33 → [-1, 1]
    3D: range 21.46 → [-1, 1]
✓ Wrote 1D positions (3,652,086 cells × 1 dims) to /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/points_1d.bin.gz (gzip)
✓ Wrote 2D positions (3,652,086 cells × 2 dims) to /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/points_2d.bin.gz (gzip)
✓ Wrote 3D positions (3,652,086 cells × 3 dims) to /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/points_3d.bin.gz (gzip)
  ⚠ outlier quantiles 'cel

Exporting genes: 100%|██████████| 12288/12288 [33:36<00:00,  6.09it/s] 


✓ Wrote var manifest (12288 genes, 8-bit quantized, gzip level 6) to /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/var_manifest.json
  Extracting unique edges from 3,652,086 cells...
  Found 39,104,100 unique edges, max 277 neighbors/cell
  Sorting edges for optimal compression...
✓ Wrote connectivity (39,104,100 edges, max 277 neighbors/cell, uint32) to /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/connectivity
✓ Wrote dataset identity to /ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/dataset_identity.json


## Validate export artifacts

Spot-check file sizes (MB), manifest stats, and total obs/var directory sizes.


In [17]:
import json
from pathlib import Path

BYTES_IN_MB = 1024 * 1024

def size_mb(path: Path) -> float:
    return round(path.stat().st_size / BYTES_IN_MB, 3) if path.exists() else 0

def dir_stats(path: Path) -> dict:
    if not path.exists():
        return {"size_mb": 0, "files": 0}
    total_bytes = 0
    file_count = 0
    for p in path.rglob("*"):
        if p.is_file():
            file_count += 1
            total_bytes += p.stat().st_size
    return {"size_mb": round(total_bytes / BYTES_IN_MB, 3), "files": file_count}

# Multi-dimensional point files
points_files = {
    'points_1d': EXPORT_DIR / "points_1d.bin.gz",
    'points_2d': EXPORT_DIR / "points_2d.bin.gz",
    'points_3d': EXPORT_DIR / "points_3d.bin.gz",
    'points (legacy)': EXPORT_DIR / "points.bin.gz",
}

obs_manifest_path = EXPORT_DIR / "obs_manifest.json"
var_manifest_path = EXPORT_DIR / "var_manifest.json"
dataset_identity_path = EXPORT_DIR / "dataset_identity.json"
obs_dir = EXPORT_DIR / "obs"
var_dir = EXPORT_DIR / "var"

obs_manifest = json.loads(obs_manifest_path.read_text()) if obs_manifest_path.exists() else None
var_manifest = json.loads(var_manifest_path.read_text()) if var_manifest_path.exists() else None
dataset_identity = json.loads(dataset_identity_path.read_text()) if dataset_identity_path.exists() else None

# Check which point files exist
point_sizes = {}
for name, path in points_files.items():
    if path.exists():
        point_sizes[name] = size_mb(path)
        print(f"✓ {name}: {point_sizes[name]} MB")
    else:
        print(f"✗ {name}: not found")

# Show embeddings metadata
if dataset_identity and 'embeddings' in dataset_identity:
    embeddings_meta = dataset_identity['embeddings']
    print(f"\nEmbeddings metadata:")
    print(f"  Available dimensions: {embeddings_meta.get('available_dimensions')}")
    print(f"  Default dimension: {embeddings_meta.get('default_dimension')}D")

{
    "paths": {
        "export_dir": str(EXPORT_DIR),
        "obs_manifest": str(obs_manifest_path),
        "var_manifest": str(var_manifest_path),
        "dataset_identity": str(dataset_identity_path),
        "obs_dir": str(obs_dir),
        "var_dir": str(var_dir),
    },
    "sizes_mb": {
        **point_sizes,
        "obs_manifest": size_mb(obs_manifest_path),
        "var_manifest": size_mb(var_manifest_path),
        "dataset_identity": size_mb(dataset_identity_path),
    },
    "dir_sizes_mb": {
        "obs": dir_stats(obs_dir),
        "var": dir_stats(var_dir),
    },
    "manifest_stats": {
        "obs": None if obs_manifest is None else {
            "n_points": obs_manifest.get("n_points"),
            "fields": len(obs_manifest.get("fields", [])),
            "centroid_outlier_quantile": obs_manifest.get("centroid_outlier_quantile"),
        },
        "var": None if var_manifest is None else {
            "n_points": var_manifest.get("n_points"),
            "fields": len(var_manifest.get("fields", [])),
            "var_gene_id_column": var_manifest.get("var_gene_id_column"),
        },
        "embeddings": None if dataset_identity is None else dataset_identity.get("embeddings"),
    },
}

✓ points_1d: 11.953 MB
✓ points_2d: 25.002 MB
✓ points_3d: 37.956 MB
✗ points (legacy): not found

Embeddings metadata:
  Available dimensions: [1, 2, 3]
  Default dimension: 3D


{'paths': {'export_dir': '/ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca',
  'obs_manifest': '/ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/obs_manifest.json',
  'var_manifest': '/ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/var_manifest.json',
  'dataset_identity': '/ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/dataset_identity.json',
  'obs_dir': '/ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/obs',
  'var_dir': '/ictstr01/groups/ml01/workspace/kemal.inecik/master_cellucid/cellucid-datasets/exports/hdca/var'},
 'sizes_mb': {'points_1d': 11.953,
  'points_2d': 25.002,
  'points_3d': 37.956,
  'obs_manifest': 1.23,
  'var_manifest': 0.439,
  'dataset_identity': 0.004},
 'dir_sizes_mb': {'obs': {'size_mb': 132.482, 'files': 75},
  'var': {'size_mb': 4261.114, 'fi

Done. Serve `index.html` from the repo root to view the exported data.
